## Introduction To Docker

Docker is an open-source platform that packages software and all its dependencies into a self-contained unit called a container. A container holds your code, runtime, system libraries, and configuration together  guaranteeing the application runs identically on any machine that has Docker installed, whether that is your laptop or teammate's machine



### Why Docker Is Neccesary

**Reproducibility across environments:** Docker ensures your application runs the same way on every machine by packaging the code, dependencies, and configuration together.

**Cloud portability:** Docker containers can be deployed consistently across different cloud providers without modifying the application.
Dependency isolation: Docker keeps project dependencies separate, preventing conflicts with other applications or the host system.

**Consistent CI/CD pipelines:** Docker provides a standardized environment so builds, tests, and deployments behave the same throughout the CI/CD process.

### Installing Docker 
(Skip if you have it already)

# Installing Docker on WSL (Windows)

1. Enable WSL 2 (if not already installed):
   ```bash
   wsl --install
   ```

2. Download and install Docker Desktop:
   https://docs.docker.com/desktop/install/windows-install/

3. Open **Docker Desktop** → **Settings** → **Resources** → **WSL Integration**.

4. Enable integration for your WSL distribution (e.g., Ubuntu).

5. Restart Docker Desktop and verify the installation:
   ```bash
   docker --version
   docker run hello-world
   ```

# Installing Docker on macOS

1. Download and install **Docker Desktop for Mac**:
   https://docs.docker.com/desktop/install/mac-install/

2. Open **Docker Desktop** and complete the initial setup.

3. Grant any required system permissions when prompted.

4. Verify the installation:
   ```bash
   docker --version
   docker run hello-world
   ```

# Docker Basic Command


Check docker verion
```bash
docker --version
```


Run a simple container:

```bash
docker run hello-world
```


Run something more complex:

```bash
docker run ubuntu
```

Nothing happens here. Run it in Interactive  mode (-it):

```bash
docker run -it ubuntu
```



The ubuntu has no package like python there, so lets install it:

```bash
apt update && apt install python3
python3 -V
```

## Stateless Containers

Docker containers are **stateless by default**, meaning any changes made inside a running container are temporary. Once the container is stopped and removed, all modifications are lost unless the data is stored in a Docker volume or bind mount.

For example, if you install software or create files inside a container:

```bash
docker run -it ubuntu
python3 -V
```

Those changes exist only for that container instance. If you remove the container and start a new one, it begins from the original image with none of the previous modifications.

This behavior makes containers predictable and safe to experiment with. Even if you accidentally delete system files inside a container, the next container you create from the same image starts with a clean, unchanged filesystem.

```bash
docker run -it ubuntu
rm -rf / # don't run it on your computer,it wipes every file!
```
Next time we run ```docker run -it ubuntu ```, all the files are back.


# Managing Docker Containers

Docker provides simple commands to manage running and stopped containers.

## Viewing Containers (`docker ps`)

You can list currently running containers using:

```bash
docker ps
```

To see all containers (including stopped ones), use:

```bash
docker ps -a
```

This helps you track container IDs, names, status, and which images they were created from.

---

## Removing Containers (`docker rm`)

To delete a stopped container, use:

```bash
docker rm <container_id>
```

Example:

```bash
docker rm 2f4a1c9d3b21
```

To remove multiple containers at once:

```bash
docker rm container1 container2
```
```

# Different Base Images & Docker Volumes

## Different Base Images

Docker images are not limited to just `hello-world` or `ubuntu`. You can choose different base images depending on your use case.

For example, a Python environment:

```bash
docker run -it --rm python:3.9.16
```

This starts an interactive Python environment inside the container.

If you want a smaller version of the image, you can use the **slim** tag:

```bash
docker run -it --rm python:3.9.16-slim
```

### Overriding the Default Command (Entrypoint)

Sometimes you don’t want the default Python shell—you may want a terminal instead. You can override the entrypoint:

```bash
docker run -it \
    --rm \
    --entrypoint=bash \
    python:3.9.16-slim
```

This starts a Bash shell instead of Python.

---

## Volumes (Sharing Data Between Host and Container)

Containers are temporary, but your data can be shared using **volumes**.

### Create some files on your host machine:

```bash
### Create some files on your host machine

```bash
mkdir project_data
cd project_data
touch users.txt transactions.txt config.txt
echo "Hello from the host machine" > users.txt
cd ..
```

---

### Create a Python script to read the files

Create `project_data/read_files.py`:

```python
from pathlib import Path

current_dir = Path.cwd()
current_file = Path(__file__).name

print(f"Files in {current_dir}:")

for filepath in current_dir.iterdir():
    if filepath.name == current_file:
        continue

    print(f"  - {filepath.name}")

    if filepath.is_file():
        content = filepath.read_text(encoding="utf-8")
        print(f"    Content: {content}")
```

---

### Run Docker with a mounted volume

```bash
docker run -it \
    --rm \
    -v $(pwd)/test:/app/test \
    --entrypoint=bash \
    python:3.9.16-slim
```

---

### Inside the container:

```bash
cd /app/project_data
ls -la
cat  users.txt
python read_files.py
```

You will see that the container can access files from your **host machine** through the mounted volume.

Note: If you deleted a file mounted on the container, it will also be deleted from the host machine
```